In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/cloudsentry', exist_ok=True)
os.chdir('/content/drive/MyDrive/cloudsentry')
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/cloudsentry


In [ ]:
!curl -sSfL https://raw.githubusercontent.com/anchore/syft/main/install.sh | sh -s -- -b /usr/local/bin
!syft version

[info] checking github for the current release tag 
[info] fetching release script for tag='v1.54.1' 
[info] checking github for the current release tag 
[info] using release tag='v1.54.1' version='1.54.1' os='linux' arch='amd64' 
[info] installed /usr/local/bin/syft 
Application:   syft
Version:       1.54.1
BuildDate:     2026-10-06T15:32:19Z
GitCommit:     b254e6d92f28c3868a755f62fb3ca8f26e9fee76
GitDescription: v1.54.1
Platform:      linux/amd64
GoVersion:     go1.26.8
Compiler:      gc
SchemaVersion: 16.1.11


In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor

os.makedirs("sboms/normal", exist_ok=True)
R = "registry:public.ecr.aws/docker/library/"
images = [
    "python:3.11-slim","python:3.12-slim","python:3.13-slim","python:3.12",
    "node:18-slim","node:20-slim","node:22-slim","node:22-alpine",
    "nginx:1.27","nginx:alpine","httpd:2.4","haproxy:3.0","traefik:v3.1",
    "redis:7","memcached:1.6","postgres:16","postgres:15","mysql:8.4","mongo:7","rabbitmq:3",
    "golang:1.22","golang:1.23","rust:1-slim","ruby:3.3-slim","php:8.3-apache",
    "eclipse-temurin:17-jre","eclipse-temurin:21-jre","tomcat:10","maven:3-eclipse-temurin-21",
    "ubuntu:22.04","ubuntu:24.04","debian:11-slim","debian:12-slim","alpine:3.19","alpine:3.20",
    "busybox:1.36","wordpress:6","ghost:5","nextcloud:29","elasticsearch:8.15.0",
]

def scan(img):
    out = f"sboms/normal/{img.replace(':', '__')}.json"
    if os.path.exists(out):
        return img, "cached"
    r = subprocess.run(["syft", R + img, "-o", f"syft-json={out}", "-q"],
                       capture_output=True, text=True, timeout=900)
    return img, "ok" if r.returncode == 0 else r.stderr.strip()[-150:]

with ThreadPoolExecutor(4) as ex:
    for img, status in ex.map(scan, images):
        print(f"{img:30s} {status}")

python:3.11-slim               ok
python:3.12-slim               ok
python:3.13-slim               ok
python:3.12                    ok
node:18-slim                   ok
node:20-slim                   ok
node:22-slim                   ok
node:22-alpine                 ok
nginx:1.27                     ok
nginx:alpine                   ok
httpd:2.4                      ok
haproxy:3.0                    ok
traefik:v3.1                   ok
redis:7                        ok
memcached:1.6                  ok
postgres:16                    ok
postgres:15                    ok
mysql:8.4                      ok
mongo:7                        ok
rabbitmq:3                     ok
golang:1.22                    ok
golang:1.23                    ok
rust:1-slim                    ok
ruby:3.3-slim                  ok
php:8.3-apache                 ok
eclipse-temurin:17-jre         ok
eclipse-temurin:21-jre         ok
tomcat:10                      ok
maven:3-eclipse-temurin-21     ok
ubuntu:22.04  

In [ ]:
!ls sboms/normal | wc -l && du -sh sboms/normal

40
198M	sboms/normal


In [ ]:
import json, glob, math, pickle
from collections import Counter
import pandas as pd

TYPES = ["deb","apk","rpm","python","npm","go-module","java-archive","gem","php-composer","binary"]
SUSPICIOUS = {"nmap","netcat","netcat-openbsd","netcat-traditional","ncat","socat","xmrig","masscan",
              "hydra","john","proxychains","proxychains-ng","tor","minerd","cpuminer","hping3","sshpass"}
POPULAR = {"requests","urllib3","numpy","pandas","django","flask","boto3","setuptools","cryptography",
           "pyyaml","jinja2","certifi","idna","colorama","python-dateutil","lodash","express","react",
           "axios","chalk","commander","moment","webpack","typescript","debug","colors","request",
           "openssl","libssl3","zlib1g","libcurl4","curl","bash","coreutils","sudo"}

def lev1(a, b):
    """True if a and b differ by exactly one edit."""
    if a == b or abs(len(a) - len(b)) > 1: return False
    if len(a) == len(b): return sum(x != y for x, y in zip(a, b)) == 1
    s, l = (a, b) if len(a) < len(b) else (b, a)
    return any(l[:i] + l[i+1:] == s for i in range(len(l)))

def load(path):
    arts = json.load(open(path))["artifacts"]
    return [(a.get("name","").lower(), a.get("type",""), a.get("version",""), a.get("licenses", [])) for a in arts]

normal = {p.split("/")[-1][:-5]: load(p) for p in sorted(glob.glob("sboms/normal/*.json"))}

# How many baseline images contain each (type, name) package
FREQ = Counter()
for pkgs in normal.values():
    FREQ.update({(t, n) for n, t, _, _ in pkgs})

def featurize(pkgs, exclude_self=False):
    n = max(len(pkgs), 1)
    keys = {(t, name) for name, t, _, _ in pkgs}
    tc = Counter(t for _, t, _, _ in pkgs)
    names = {name for name, _, _, _ in pkgs}
    f = {
        "log_n_pkgs": math.log1p(len(pkgs)),
        "n_ecosystems": sum(1 for t in tc if tc[t] > 0),
        "frac_no_version": sum(1 for _, _, v, _ in pkgs if not v) / n,
        "frac_no_license": sum(1 for *_, l in pkgs if not l) / n,
        # rare = seen in no OTHER baseline image (leave-one-out for training images)
        "rare_ratio": sum(1 for k in keys if FREQ[k] - (1 if exclude_self else 0) == 0) / max(len(keys), 1),
        "n_suspicious": len(names & SUSPICIOUS),
        "n_typosquat": sum(1 for nm in names if len(nm) >= 5 and nm not in POPULAR
                           and any(lev1(nm, p) for p in POPULAR)),
    }
    for t in TYPES:
        f[f"frac_{t}"] = tc[t] / n
    return f

feats = pd.DataFrame({img: featurize(p, exclude_self=True) for img, p in normal.items()}).T
feats.to_parquet("features_normal.parquet")
pickle.dump({"FREQ": FREQ, "n_baseline": len(normal)}, open("baseline_freq.pkl", "wb"))

pd.set_option("display.width", 200)
print(feats[["log_n_pkgs","n_ecosystems","rare_ratio","n_suspicious","n_typosquat","frac_no_license"]].round(3))

                             log_n_pkgs  n_ecosystems  rare_ratio  n_suspicious  n_typosquat  frac_no_license
alpine__3.19                      2.773           1.0       0.067           0.0          0.0            0.000
alpine__3.20                      2.708           1.0       0.000           0.0          0.0            0.000
busybox__1.36                     0.693           1.0       1.000           0.0          0.0            1.000
debian__11-slim                   4.575           1.0       0.010           0.0          0.0            0.000
debian__12-slim                   4.489           1.0       0.000           0.0          0.0            0.000
eclipse-temurin__17-jre           5.050           4.0       0.000           0.0          0.0            0.090
eclipse-temurin__21-jre           5.050           4.0       0.000           0.0          0.0            0.090
elasticsearch__8.15.0             6.615           3.0       0.784           2.0          0.0            0.260
ghost__5  

In [ ]:
for img in ["elasticsearch__8.15.0", "haproxy__3.0", "ghost__5", "mysql__8.4", "nginx__alpine"]:
    pkgs = normal[img]
    names = {(n, t) for n, t, _, _ in pkgs}
    sus = [(n, t) for n, t in names if n in SUSPICIOUS]
    typo = [(n, t, [p for p in POPULAR if lev1(n, p)]) for n, t in names
            if len(n) >= 5 and n not in POPULAR and any(lev1(n, p) for p in POPULAR)]
    print(img, "| suspicious:", sus, "| typosquat:", typo)

elasticsearch__8.15.0 | suspicious: [('netcat', 'deb'), ('netcat-openbsd', 'deb')] | typosquat: []
haproxy__3.0 | suspicious: [('socat', 'deb')] | typosquat: []
ghost__5 | suspicious: [] | typosquat: [('color', 'npm', ['colors'])]
mysql__8.4 | suspicious: [] | typosquat: [('libcurl', 'rpm', ['libcurl4'])]
nginx__alpine | suspicious: [] | typosquat: [('libcurl', 'apk', ['libcurl4'])]


In [ ]:
# Real attack tools vs. legit-but-abusable admin tools
OFFENSIVE = {"nmap","xmrig","masscan","hydra","john","minerd","cpuminer","hping3",
             "proxychains","proxychains-ng","sshpass"}
DUAL_USE = {"netcat","netcat-openbsd","netcat-traditional","ncat","socat","tor"}

# Typosquatting only happens in public language registries
REGISTRY_TYPES = {"python","npm","gem"}
POPULAR = {"requests","urllib3","numpy","pandas","django","flask","boto3","setuptools","cryptography",
           "pyyaml","jinja2","certifi","idna","colorama","python-dateutil","lodash","express","react",
           "axios","chalk","commander","moment","webpack","typescript","debug","colors","request"}
KNOWN_LEGIT = {"color"}  # real packages that happen to be 1 edit from a popular name

def featurize(pkgs, exclude_self=False):
    n = max(len(pkgs), 1)
    keys = {(t, name) for name, t, _, _ in pkgs}
    tc = Counter(t for _, t, _, _ in pkgs)
    names = {name for name, _, _, _ in pkgs}
    reg_names = {name for name, t, _, _ in pkgs if t in REGISTRY_TYPES}
    f = {
        "log_n_pkgs": math.log1p(len(pkgs)),
        "n_ecosystems": sum(1 for t in tc if tc[t] > 0),
        "frac_no_version": sum(1 for _, _, v, _ in pkgs if not v) / n,
        "frac_no_license": sum(1 for *_, l in pkgs if not l) / n,
        "rare_ratio": sum(1 for k in keys if FREQ[k] - (1 if exclude_self else 0) == 0) / max(len(keys), 1),
        "n_offensive": len(names & OFFENSIVE),
        "n_dual_use": len(names & DUAL_USE),
        "n_typosquat": sum(1 for nm in reg_names if len(nm) >= 5 and nm not in POPULAR
                           and nm not in KNOWN_LEGIT and any(lev1(nm, p) for p in POPULAR)),
    }
    for t in TYPES:
        f[f"frac_{t}"] = tc[t] / n
    return f

feats = pd.DataFrame({img: featurize(p, exclude_self=True) for img, p in normal.items()}).T
feats.to_parquet("features_normal.parquet")
print(feats[["n_offensive","n_dual_use","n_typosquat"]].sum())
print(feats[feats["n_dual_use"] > 0][["n_dual_use"]])

n_offensive    0.0
n_dual_use     3.0
n_typosquat    0.0
dtype: float64
                       n_dual_use
elasticsearch__8.15.0         2.0
haproxy__3.0                  1.0


In [ ]:
import numpy as np, random
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

IF_COLS = [c for c in feats.columns if c not in ("n_offensive", "n_typosquat")]
random.seed(0)

def rnd(prefix): return f"{prefix}-{random.getrandbits(32):08x}"

def inject(pkgs, attack, k):
    if attack == "foreign_ecosystem":   # e.g. npm packages suddenly inside a python image
        t = "python" if any(p[1] == "npm" for p in pkgs) else "npm"
        extra = [(rnd("lib"), t, "1.0.0", ["MIT"]) for _ in range(k)]
    elif attack == "unknown_binaries":  # unversioned, unlicensed binaries dropped in
        extra = [(rnd("bin"), "binary", "", []) for _ in range(k)]
    elif attack == "unvetted_deps":     # never-seen deps from the image's own registry, no license
        t = next((p[1] for p in pkgs if p[1] in REGISTRY_TYPES), "python")
        extra = [(rnd("dep"), t, "0.0.1", []) for _ in range(k)]
    return pkgs + extra

def rules(f):
    return f["n_offensive"] > 0 or f["n_typosquat"] > 0

# --- Leave-one-out evaluation ---
imgs = list(normal)
X_norm = feats.loc[imgs, IF_COLS].values.astype(float)
rows = []
for i, img in enumerate(imgs):
    iso = IsolationForest(n_estimators=300, random_state=0).fit(np.delete(X_norm, i, axis=0))
    rows.append({"attack": "none", "k": 0, "score": -iso.score_samples(X_norm[[i]])[0]})
    for attack in ["foreign_ecosystem", "unknown_binaries", "unvetted_deps"]:
        for k in [5, 20, 50]:
            f = featurize(inject(normal[img], attack, k), exclude_self=True)
            x = np.array([[f[c] for c in IF_COLS]], dtype=float)
            rows.append({"attack": attack, "k": k, "score": -iso.score_samples(x)[0]})
res = pd.DataFrame(rows)

norm_scores = res[res.attack == "none"].score
thr = norm_scores.quantile(0.95)
print(f"threshold = {thr:.3f} | false positive rate on normal images: {(norm_scores > thr).mean():.3f}\n")

out = []
for (attack, k), g in res[res.attack != "none"].groupby(["attack", "k"]):
    auc = roc_auc_score([0]*len(norm_scores) + [1]*len(g), list(norm_scores) + list(g.score))
    out.append({"attack": attack, "k": k, "detect_rate": (g.score > thr).mean(), "auc": auc})
print(pd.DataFrame(out).round(3).to_string(index=False))

# --- Rule layer check ---
print()
base = normal["python__3.12-slim"]
for name, extra in [("cryptominer", [("xmrig", "binary", "6.21", [])]),
                    ("typosquat",   [("reqests", "python", "2.31.0", ["MIT"])])]:
    print(f"{name:12s} → rule flag: {rules(featurize(base + extra, exclude_self=True))}")

threshold = 0.561 | false positive rate on normal images: 0.050

           attack  k  detect_rate   auc
foreign_ecosystem  5        0.050 0.594
foreign_ecosystem 20        0.050 0.618
foreign_ecosystem 50        0.050 0.639
 unknown_binaries  5        0.025 0.579
 unknown_binaries 20        0.050 0.689
 unknown_binaries 50        0.050 0.748
    unvetted_deps  5        0.075 0.632
    unvetted_deps 20        0.075 0.761
    unvetted_deps 50        0.050 0.787

cryptominer  → rule flag: True
typosquat    → rule flag: True


In [ ]:
os.makedirs("sboms/versions", exist_ok=True)
series = {
    "python": [f"3.12.{i}-slim" for i in range(1, 8)],
    "node":   [f"20.{i}.0-slim" for i in range(10, 19)],
    "nginx":  [f"1.25.{i}" for i in range(0, 6)],
    "golang": [f"1.22.{i}" for i in range(0, 9)],
    "postgres": [f"16.{i}" for i in range(0, 5)],
    "redis":  [f"7.2.{i}" for i in range(0, 7)],
    "ruby":   [f"3.3.{i}-slim" for i in range(0, 6)],
}
ver_images = [f"{repo}:{tag}" for repo, tags in series.items() for tag in tags]

def scan_v(img):
    out = f"sboms/versions/{img.replace(':', '__')}.json"
    if os.path.exists(out): return img, "cached"
    r = subprocess.run(["syft", R + img, "-o", f"syft-json={out}", "-q"],
                       capture_output=True, text=True, timeout=900)
    return img, "ok" if r.returncode == 0 else r.stderr.strip()[-120:]

with ThreadPoolExecutor(4) as ex:
    for img, status in ex.map(scan_v, ver_images):
        print(f"{img:25s} {status}")

python:3.12.1-slim        ok
python:3.12.2-slim        ok
python:3.12.3-slim        ok
python:3.12.4-slim        ok
python:3.12.5-slim        ok
python:3.12.6-slim        ok
python:3.12.7-slim        ok
node:20.10.0-slim         ok
node:20.11.0-slim         ok
node:20.12.0-slim         ok
node:20.13.0-slim         ok
node:20.14.0-slim         ok
node:20.15.0-slim         ok
node:20.16.0-slim         ok
node:20.17.0-slim         ok
node:20.18.0-slim         ok
nginx:1.25.0              ok
nginx:1.25.1              ok
nginx:1.25.2              ok
nginx:1.25.3              ok
nginx:1.25.4              ok
nginx:1.25.5              ok
golang:1.22.0             ok
golang:1.22.1             ok
golang:1.22.2             ok
golang:1.22.3             ok
golang:1.22.4             ok
golang:1.22.5             ok
golang:1.22.6             ok
golang:1.22.7             ok
golang:1.22.8             ok
postgres:16.0             ok
postgres:16.1             ok
postgres:16.2             ok
postgres:16.3 

In [ ]:
# --- Load version SBOMs and build consecutive pairs ---
ver = {p.split("/")[-1][:-5]: load(p) for p in sorted(glob.glob("sboms/versions/*.json"))}
pairs = []
for repo, tags in series.items():
    keys = [f"{repo}__{t}" for t in tags if f"{repo}__{t}" in ver]
    pairs += [(repo, keys[i], keys[i+1]) for i in range(len(keys) - 1)]
print(len(pairs), "normal update pairs")

# Corpus-wide package frequency (all scanned images)
GFREQ = Counter()
for pkgs in list(normal.values()) + list(ver.values()):
    GFREQ.update({(t, n) for n, t, _, _ in pkgs})

def as_dict(pkgs):
    return {(t, n): (v, l) for n, t, v, l in pkgs}

def drift(prev, new):
    p, n = as_dict(prev), as_dict(new)
    added = [k for k in n if k not in p]
    removed = [k for k in p if k not in n]
    a = max(len(added), 1)
    return {
        "log_added": math.log1p(len(added)),
        "log_removed": math.log1p(len(removed)),
        "frac_added": len(added) / max(len(p), 1),
        "size_log_ratio": math.log((len(n) + 1) / (len(p) + 1)),
        "n_new_ecosystems": len({t for t, _ in n} - {t for t, _ in p}),
        "added_no_license": sum(1 for k in added if not n[k][1]) / a,
        "added_no_version": sum(1 for k in added if not n[k][0]) / a,
        "added_binary": sum(1 for k in added if k[0] == "binary") / a,
        "added_rare": sum(1 for k in added if GFREQ[k] - 1 <= 0) / a,  # -1: exclude the new image itself
    }

D_COLS = list(drift(ver[pairs[0][1]], ver[pairs[0][2]]).keys())
normal_pairs = pd.DataFrame([{"repo": r, **drift(ver[a], ver[b])} for r, a, b in pairs])

# --- Leave-one-series-out evaluation ---
rows = []
for repo in series:
    train = normal_pairs[normal_pairs.repo != repo][D_COLS].values
    test_pairs = [(a, b) for r, a, b in pairs if r == repo]
    if not test_pairs: continue
    iso = IsolationForest(n_estimators=300, random_state=0).fit(train)
    for a, b in test_pairs:
        x = np.array([list(drift(ver[a], ver[b]).values())])
        rows.append({"attack": "none", "k": 0, "score": -iso.score_samples(x)[0],
                     "log_added": x[0][0]})
        for attack in ["foreign_ecosystem", "unknown_binaries", "unvetted_deps"]:
            for k in [5, 20, 50]:
                x = np.array([list(drift(ver[a], inject(ver[b], attack, k)).values())])
                rows.append({"attack": attack, "k": k, "score": -iso.score_samples(x)[0],
                             "log_added": x[0][0]})
res = pd.DataFrame(rows)

norm = res[res.attack == "none"]
thr = norm.score.quantile(0.95)
print(f"threshold = {thr:.3f} | false positive rate: {(norm.score > thr).mean():.3f}\n")

out = []
for (attack, k), g in res[res.attack != "none"].groupby(["attack", "k"]):
    y = [0]*len(norm) + [1]*len(g)
    out.append({"attack": attack, "k": k,
                "detect_rate": (g.score > thr).mean(),
                "auc_iforest": roc_auc_score(y, list(norm.score) + list(g.score)),
                "auc_count_only": roc_auc_score(y, list(norm.log_added) + list(g.log_added))})
print(pd.DataFrame(out).round(3).to_string(index=False))

42 normal update pairs
threshold = 0.658 | false positive rate: 0.071

           attack  k  detect_rate  auc_iforest  auc_count_only
foreign_ecosystem  5        0.738        0.949           0.946
foreign_ecosystem 20        1.000        0.975           0.955
foreign_ecosystem 50        1.000        0.977           1.000
 unknown_binaries  5        1.000        0.977           0.946
 unknown_binaries 20        1.000        0.985           0.955
 unknown_binaries 50        1.000        0.995           1.000
    unvetted_deps  5        1.000        0.977           0.946
    unvetted_deps 20        1.000        0.980           0.955
    unvetted_deps 50        1.000        0.980           1.000


In [ ]:
def stealth(pkgs, attack, k):
    if attack == "swap_binaries":
        keep = random.sample(pkgs, max(len(pkgs) - k, 0))
        return keep + [(rnd("bin"), "binary", "", []) for _ in range(k)]
    return inject(pkgs, attack, k)

ATTACKS = ["foreign_ecosystem", "unknown_binaries", "unvetted_deps", "swap_binaries"]
rows = []
for repo in series:
    train = normal_pairs[normal_pairs.repo != repo][D_COLS].values
    test_pairs = [(a, b) for r, a, b in pairs if r == repo]
    if not test_pairs: continue
    iso = IsolationForest(n_estimators=300, random_state=0).fit(train)
    for a, b in test_pairs:
        x = np.array([list(drift(ver[a], ver[b]).values())])
        rows.append({"attack": "none", "k": 0, "score": -iso.score_samples(x)[0], "log_added": x[0][0]})
        for attack in ATTACKS:
            for k in [1, 2, 3]:
                x = np.array([list(drift(ver[a], stealth(ver[b], attack, k)).values())])
                rows.append({"attack": attack, "k": k, "score": -iso.score_samples(x)[0], "log_added": x[0][0]})
res = pd.DataFrame(rows)

norm = res[res.attack == "none"]
thr = norm.score.quantile(0.95)
print(f"threshold = {thr:.3f} | false positive rate: {(norm.score > thr).mean():.3f}\n")
out = []
for (attack, k), g in res[res.attack != "none"].groupby(["attack", "k"]):
    y = [0]*len(norm) + [1]*len(g)
    out.append({"attack": attack, "k": k,
                "detect_rate": (g.score > thr).mean(),
                "auc_iforest": roc_auc_score(y, list(norm.score) + list(g.score)),
                "auc_count_only": roc_auc_score(y, list(norm.log_added) + list(g.log_added))})
print(pd.DataFrame(out).round(3).to_string(index=False))

threshold = 0.658 | false positive rate: 0.071

           attack  k  detect_rate  auc_iforest  auc_count_only
foreign_ecosystem  1        0.214        0.875           0.841
foreign_ecosystem  2        0.405        0.931           0.915
foreign_ecosystem  3        0.524        0.944           0.933
    swap_binaries  1        1.000        0.975           0.841
    swap_binaries  2        1.000        0.977           0.915
    swap_binaries  3        1.000        0.977           0.933
 unknown_binaries  1        0.976        0.976           0.841
 unknown_binaries  2        1.000        0.977           0.915
 unknown_binaries  3        1.000        0.977           0.933
    unvetted_deps  1        0.952        0.964           0.841
    unvetted_deps  2        1.000        0.974           0.915
    unvetted_deps  3        1.000        0.976           0.933


In [ ]:
import joblib, sklearn

final_iso = IsolationForest(n_estimators=300, random_state=0).fit(normal_pairs[D_COLS].values)
joblib.dump({
    "model": final_iso,
    "cols": D_COLS,
    "threshold": float(thr),          # from out-of-sample scores, not training scores
    "gfreq": dict(GFREQ),
    "offensive": sorted(OFFENSIVE), "dual_use": sorted(DUAL_USE),
    "popular": sorted(POPULAR), "known_legit": sorted(KNOWN_LEGIT),
    "registry_types": sorted(REGISTRY_TYPES),
}, "sbom_model.joblib")
print("sklearn", sklearn.__version__)
!ls -lh sbom_model.joblib

sklearn 1.6.1
-rw------- 1 root root 863K Oct 12 19:37 sbom_model.joblib
